# K-Nearest Neighbors Training with Polars

This notebook keeps the flow simple:
- load the prepared parquet files
- separate features and target
- validate the numeric training inputs
- train and evaluate the model
- save the model and CSV outputs


## 1. Imports and Config


In [8]:
from pathlib import Path

import polars as pl
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
from sklearn.neighbors import KNeighborsClassifier
import joblib

TRAIN_FILE = "../data/processed/train.parquet"
TEST_FILE = "../data/processed/test.parquet"
MODEL_DIR = "../model"
RESULTS_DIR = "../model/results"
TARGET_COLUMN = "admitted"
RANDOM_SEED = 42
MODEL_NAME = "knn"

Path(MODEL_DIR).mkdir(parents=True, exist_ok=True)
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

MODEL_FILE = Path(MODEL_DIR) / "knn_model.joblib"
RESULTS_FILE = Path(RESULTS_DIR) / "knn_results.csv"
METRICS_FILE = Path(RESULTS_DIR) / "knn_metrics.csv"


## 2. Load the Dataset


In [9]:
for required_file in [TRAIN_FILE, TEST_FILE]:
    if not Path(required_file).exists():
        raise FileNotFoundError(f"Expected parquet file at: {required_file}")

train_df = pl.read_parquet(TRAIN_FILE)
test_df = pl.read_parquet(TEST_FILE)

print(f"train shape: {train_df.shape}")
print(f"test shape: {test_df.shape}")
train_df.head()


train shape: (56, 8)
test shape: (16, 8)


age,length_of_stay,prior_admissions,lab_score,comorbidity_count,medications_count,followup_gap_days,admitted
i64,i64,i64,f64,i64,i64,i64,i64
34,2,4,0.47,2,6,21,1
35,3,3,0.65,1,2,19,1
71,7,0,0.41,1,5,3,1
72,7,1,0.53,5,3,15,1
49,8,2,0.65,2,9,17,1


## 3. Separate Features and Target

This notebook uses the prepared train and test parquet files exactly as they come from the upstream split notebook.


In [10]:
if TARGET_COLUMN not in train_df.columns or TARGET_COLUMN not in test_df.columns:
    raise ValueError(f"Both parquet files must include the target column: {TARGET_COLUMN}")

feature_columns = [column_name for column_name in train_df.columns if column_name != TARGET_COLUMN]

X_train = train_df.select(feature_columns)
X_test = test_df.select([column_name for column_name in test_df.columns if column_name != TARGET_COLUMN])
y_train = train_df[TARGET_COLUMN].cast(pl.Int32)
y_test = test_df[TARGET_COLUMN].cast(pl.Int32)

print(feature_columns)
print(X_train.shape)
X_train.head()


['age', 'length_of_stay', 'prior_admissions', 'lab_score', 'comorbidity_count', 'medications_count', 'followup_gap_days']
(56, 7)


age,length_of_stay,prior_admissions,lab_score,comorbidity_count,medications_count,followup_gap_days
i64,i64,i64,f64,i64,i64,i64
34,2,4,0.47,2,6,21
35,3,3,0.65,1,2,19
71,7,0,0.41,1,5,3
72,7,1,0.53,5,3,15
49,8,2,0.65,2,9,17


## 4. Validate the Prepared Features

These notebooks expect the upstream split and feature-selection notebook to output numeric, model-ready feature columns.


In [11]:
test_feature_columns = [column_name for column_name in test_df.columns if column_name != TARGET_COLUMN]

if feature_columns != test_feature_columns:
    raise ValueError(
        "Train and test parquet files must have the same feature columns in the same order."
    )

non_numeric_columns = sorted(
    set(
        [column_name for column_name, dtype in X_train.schema.items() if not dtype.is_numeric()]
        + [column_name for column_name, dtype in X_test.schema.items() if not dtype.is_numeric()]
    )
)

if non_numeric_columns:
    raise ValueError(
        "The split/feature-selection notebook must output model-ready numeric features. "
        f"Non-numeric columns found: {non_numeric_columns}"
    )

print(f"validated {len(feature_columns)} numeric feature columns")


validated 7 numeric feature columns


## 5. Train the Model


In [12]:
model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train.to_numpy(), y_train.to_numpy())

model


,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.Doesn't affect :meth:`fit` method.",None


## 6. Evaluate and Save


In [13]:
probabilities = model.predict_proba(X_test.to_numpy())[:, 1]
predictions = (probabilities >= 0.5).astype(int)
actuals = y_test.to_numpy()

metrics = {
    "model": MODEL_NAME,
    "accuracy": accuracy_score(actuals, predictions),
    "precision": precision_score(actuals, predictions, zero_division=0),
    "recall": recall_score(actuals, predictions, zero_division=0),
    "roc_auc": roc_auc_score(actuals, probabilities),
}

metrics_frame = pl.DataFrame([metrics])
results_frame = test_df.with_columns(
    [
        pl.Series("actual", actuals),
        pl.Series("predicted", predictions),
        pl.Series("probability", probabilities),
        pl.lit(MODEL_NAME).alias("model"),
    ]
)

joblib.dump(model, MODEL_FILE)
metrics_frame.write_csv(METRICS_FILE)
results_frame.write_csv(RESULTS_FILE)

print(f"model saved to: {MODEL_FILE}")
print(f"metrics saved to: {METRICS_FILE}")
print(f"results saved to: {RESULTS_FILE}")
metrics_frame


model saved to: ..\model\knn_model.joblib
metrics saved to: ..\model\results\knn_metrics.csv
results saved to: ..\model\results\knn_results.csv


model,accuracy,precision,recall,roc_auc
str,f64,f64,f64,f64
"""knn""",0.9375,0.9375,1.0,0.9


## 7. Preview Results


In [14]:
results_frame.head(10)


age,length_of_stay,prior_admissions,lab_score,comorbidity_count,medications_count,followup_gap_days,admitted,actual,predicted,probability,model
i64,i64,i64,f64,i64,i64,i64,i64,i32,i64,f64,str
48,7,3,0.47,3,4,19,1,1,1,1.0,"""knn"""
30,1,2,0.59,3,7,7,1,1,1,0.8,"""knn"""
53,8,1,0.47,6,10,5,1,1,1,1.0,"""knn"""
35,2,0,0.59,6,4,13,1,1,1,0.8,"""knn"""
43,6,0,0.47,6,7,13,1,1,1,1.0,"""knn"""
31,2,1,0.77,2,3,5,0,0,1,0.8,"""knn"""
34,3,2,0.53,3,4,7,1,1,1,1.0,"""knn"""
49,7,4,0.59,1,2,11,1,1,1,1.0,"""knn"""
61,4,1,0.35,6,4,5,1,1,1,1.0,"""knn"""
